# 🗺️ Mapas y Tablas Hash - Map ADT e Implementación con Lista No Ordenada

## Programación III - Lic. en Sistemas

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap10/01_Maps_ADT_HashTables_Teoria.ipynb)

### 🎯 Objetivos

Al finalizar este notebook el estudiante podrá:
- Definir el ADT Mapa y sus operaciones según `MutableMapping` de Python
- Implementar la clase base `MapBase` con el _Item interno
- Implementar `UnsortedTableMap` y analizar su complejidad O(n)
- Aplicar mapas para contar frecuencias de palabras

### 📚 Contenido

📚 *Data Structures and Algorithms in Python* — Goodrich, Tamassia, Goldwasser (Cap. 10 Secc. 1)  

| Sección | Tema |
|---------|------|
| 10.1 | Maps and Dictionaries |
| 10.1.1 | The Map ADT |
| 10.1.2 | Application: Counting Word Frequencies |
| 10.1.3 | Python's MutableMapping ABC |
| 10.1.4 | Our MapBase Class |
| 10.1.5 | Simple Unsorted Map Implementation |

---

## 📖 ¿Qué es un Mapa (Map)?

Un **Mapa** es una colección de pares **(clave, valor)** donde:
- Cada **clave** es **única** en el mapa
- Cada clave se **asocia a exactamente un valor**
- También llamado: diccionario, tabla de símbolos, tabla asociativa

### 🌍 Analogía del Mundo Real

| Contexto | Clave | Valor |
|----------|-------|-------|
| Directorio telefónico | Nombre | Número |
| DNS | Nombre de dominio | IP |
| Agenda | Fecha | Cita |
| Dict Python | Objeto inmutable | Cualquier objeto |
| Base de datos | Clave primaria | Registro |

---

## 📋 Tabla 10.1 - ADT Mapa: Operaciones

| Operación | Descripción | Complejidad (hash) |
|-----------|-------------|-------------------|
| `M[k]` → `__getitem__(k)` | Devuelve valor asociado a `k`; `KeyError` si no existe | O(1) esperado |
| `M[k] = v` → `__setitem__(k,v)` | Asocia `k` con valor `v`, insertando o actualizando | O(1) esperado |
| `del M[k]` → `__delitem__(k)` | Elimina la entrada con clave `k`; `KeyError` si no existe | O(1) esperado |
| `len(M)` → `__len__()` | Número de entradas en el mapa | O(1) |
| `k in M` → `__contains__(k)` | `True` si el mapa contiene clave `k` | O(1) esperado |
| `M.get(k, d=None)` | Devuelve `M[k]` si existe, sino `d` | O(1) esperado |
| `M.setdefault(k, d)` | Si `k` en M retorna `M[k]`, sino setea `M[k]=d` y retorna `d` | O(1) esperado |
| `M.pop(k, d)` | Elimina y retorna `M[k]`, o `d` si no existe | O(1) esperado |
| `M.popitem()` | Elimina y retorna par `(k, v)` arbitrario | O(1) esperado |
| `M.clear()` | Elimina todas las entradas del mapa | O(n) |
| `M.keys()` | Vista de todas las claves | O(1) |
| `M.values()` | Vista de todos los valores | O(1) |
| `M.items()` | Vista de todos los pares `(k, v)` | O(1) |
| `M.update(M2)` | Asigna `M[k] = v` para cada `(k,v)` en M2 | O(len(M2)) |
| `M == N` | `True` si los mapas tienen las mismas entradas | O(n) |
| `iter(M)` → `__iter__()` | Itera sobre las claves del mapa | O(n) |


In [ ]:
# ============================================================
# MapBase: Clase base abstracta (Código 10.2 del libro)
# ============================================================
from collections.abc import MutableMapping

class MapBase(MutableMapping):
    """Clase base abstracta para mapas; extiende MutableMapping.
    
    Hereda de MutableMapping e implementa métodos derivados.
    Solo requiere que las subclases implementen:
    __getitem__, __setitem__, __delitem__, __len__, __iter__
    """
    
    # ---- Clase interna para pares clave-valor ----
    class _Item:
        """Clase ligera no-pública para almacenar pares clave-valor."""
        __slots__ = '_key', '_value'
        
        def __init__(self, k, v):
            self._key = k
            self._value = v
        
        def __eq__(self, other):
            return self._key == other._key   # comparar items por clave
        
        def __ne__(self, other):
            return not (self == other)
        
        def __lt__(self, other):
            return self._key < other._key    # comparar items por clave
        
        def __repr__(self):
            return f'({self._key!r}: {self._value!r})'

# ============================================================
# UnsortedTableMap: Implementación con lista no ordenada (Código 10.1)
# ============================================================

class UnsortedTableMap(MapBase):
    """Implementación de Map usando lista no ordenada.
    
    Análisis de complejidad:
    - __getitem__:  O(n)  — búsqueda lineal
    - __setitem__:  O(n)  — búsqueda + inserción
    - __delitem__:  O(n)  — búsqueda + eliminación
    - __len__:      O(1)
    - __iter__:     O(n)
    """
    
    def __init__(self):
        """Crea un mapa vacío."""
        self._table = []   # lista de items _Item
    
    def __getitem__(self, k):
        """Retorna valor asociado a clave k; KeyError si no existe."""
        for item in self._table:
            if k == item._key:
                return item._value
        raise KeyError(repr(k))
    
    def __setitem__(self, k, v):
        """Asigna valor v a clave k, actualizando si ya existe."""
        for item in self._table:
            if k == item._key:
                item._value = v        # clave encontrada → actualizar
                return
        self._table.append(self._Item(k, v))   # no existe → insertar al final
    
    def __delitem__(self, k):
        """Elimina entrada con clave k; KeyError si no existe."""
        for j, item in enumerate(self._table):
            if k == item._key:
                self._table.pop(j)    # eliminar entrada en posición j
                return
        raise KeyError(repr(k))
    
    def __len__(self):
        return len(self._table)
    
    def __iter__(self):
        """Genera iteración sobre todas las claves."""
        for item in self._table:
            yield item._key
    
    def __repr__(self):
        items = ', '.join(f'{k!r}: {self[k]!r}' for k in self)
        return '{' + items + '}'


# ============================================================
# Demo básico: Tabla 10.1 del libro
# ============================================================
print("=== Demo UnsortedTableMap ===")
m = UnsortedTableMap()

# Insertar entradas
m['K'] = 2;   print(f"m['K'] = 2        → {m}")
m['B'] = 4;   print(f"m['B'] = 4        → {m}")
m['U'] = 2;   print(f"m['U'] = 2        → {m}")
m['V'] = 8;   print(f"m['V'] = 8        → {m}")
m['K'] = 9;   print(f"m['K'] = 9 (upd)  → {m}")

print()
print(f"m['B']             = {m['B']}")
print(f"m.get('F', 5)      = {m.get('F', 5)}")
print(f"len(m)             = {len(m)}")

del m['V'];   print(f"del m['V']        → {m}")

print(f"\nClaves:   {list(m.keys())}")
print(f"Valores:  {list(m.values())}")
print(f"Items:    {list(m.items())}")


## 🌟 Aplicación: Conteo de Frecuencia de Palabras

Una de las aplicaciones clásicas de los mapas es **contar la frecuencia** de elementos.
El patrón `freq[word] = 1 + freq.get(word, 0)` es idiomático en Python.

### Comparación de Enfoques

| Enfoque | Ventaja | Desventaja |
|---------|---------|------------|
| `UnsortedTableMap` | Implementación clara | O(n) por operación |
| `dict` Python | O(1) esperado, optimizado | Menos transparente |
| `Counter` de collections | Más expresivo | Especializado |


In [ ]:
# ============================================================
# Aplicación: Conteo de frecuencia de palabras
# Código 10.1 del libro - Sección 10.1.2
# ============================================================

def word_frequency_unsorted(text):
    """Cuenta frecuencia de palabras usando UnsortedTableMap.
    Complejidad: O(n²) donde n es número de palabras distintas.
    """
    freq = UnsortedTableMap()
    for word in text.lower().split():
        word = word.strip('.,!?;:')
        freq[word] = 1 + freq.get(word, 0)
    return freq

def word_frequency_dict(text):
    """Cuenta frecuencia de palabras usando dict de Python.
    Complejidad: O(n) esperado.
    """
    freq = {}
    for word in text.lower().split():
        word = word.strip('.,!?;:')
        freq[word] = 1 + freq.get(word, 0)
    return freq

from collections import Counter

texto = """To be or not to be that is the question
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune
or to take arms against a sea of troubles"""

print("=== Frecuencia de Palabras (Shakespeare) ===")
freq = word_frequency_dict(texto)
top5 = sorted(freq.items(), key=lambda x: -x[1])[:5]
print("Top 5 palabras más frecuentes:")
for word, count in top5:
    bar = '█' * count
    print(f"  {word:12s} {count:3d}  {bar}")

print()
print("=== Con Counter (más pythonico) ===")
words = [w.strip('.,!?;:') for w in texto.lower().split()]
c = Counter(words)
print("Top 5 con Counter:")
for word, count in c.most_common(5):
    print(f"  {word:12s}: {count}")

print()
print("=== setdefault / defaultdict ===")
from collections import defaultdict

# Agrupar palabras por longitud
by_length = defaultdict(list)
for word in set(words):
    by_length[len(word)].append(word)

for length in sorted(by_length.keys()):
    print(f"  Longitud {length}: {sorted(by_length[length])}")


## 📊 Análisis de Complejidad: UnsortedTableMap vs dict

### Tabla Comparativa

| Operación | UnsortedTableMap | dict (Hash) |
|-----------|-----------------|-------------|
| `__getitem__` | O(n) | O(1) esperado |
| `__setitem__` | O(n) | O(1) esperado |
| `__delitem__` | O(n) | O(1) esperado |
| `__iter__` | O(n) | O(n) |
| Espacio | O(n) | O(n) |

### ¿Por qué usar UnsortedTableMap?
- **Pedagógico**: base clara para entender la interfaz Map ADT
- **Funcional para n pequeño**: lista es simple y correcta
- **Sin requisito de hashabilidad**: las claves no necesitan ser hasheables

### ¿Cuándo prefer dict?
- Siempre que sea posible para rendimiento O(1) esperado
- Las claves deben ser **hasheables** (immutables: int, str, tuple, frozenset)


In [ ]:
# ============================================================
# Benchmark: UnsortedTableMap vs dict
# ============================================================
import time, random, string

def benchmark_map(map_class, n, ops):
    """Benchmark de n inserciones + ops búsquedas aleatorias."""
    m = map_class()
    keys = [random.randint(0, n*10) for _ in range(n)]
    
    t0 = time.perf_counter()
    for k in keys:
        m[k] = k * 2
    t_insert = (time.perf_counter() - t0) * 1000
    
    t0 = time.perf_counter()
    for _ in range(ops):
        k = random.choice(keys)
        _ = m.get(k)
    t_search = (time.perf_counter() - t0) * 1000
    
    return t_insert, t_search

print(f"{'N':>6} | {'UnsortedTM insert':>18} | {'dict insert':>12} | {'Speedup':>8}")
print("-" * 60)
for n in [50, 200, 500]:
    t_u_ins, _ = benchmark_map(UnsortedTableMap, n, n)
    t_d_ins, _ = benchmark_map(dict, n, n)
    speedup = t_u_ins / max(t_d_ins, 0.001)
    print(f"{n:>6} | {t_u_ins:>16.2f}ms | {t_d_ins:>10.2f}ms | {speedup:>7.1f}x")

print("\n💡 dict es significativamente más rápido gracias a hash tables O(1)")


---

🔗 **Referencia:** Goodrich et al., Cap. 10.1

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>